## 1.使用arg_schema

### 初始化模型

In [4]:
## 1.初始化模型
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI


# 创建模型实例
model = ChatOpenAI(
    model="qwen3-vl:latest",
    api_key="sk12345",
    base_url="http://localhost:11434/v1",
    temperature=0.1
)   

### 定义工具

In [7]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool

class WeatherSchema(BaseModel):
    city:str = Field(
        description="具体的城市",
        default="北京"
    )
    if_forecast:bool = Field(
        description="是否包含明天的天气预报",
        default=False
    )


@tool(description="查询当天的天气，可以包含明天的天气预报",args_schema=WeatherSchema)
def get_weather(city:str,if_forecast:bool):
    res = f"{city}明天天气不错"
    if if_forecast:
        res += f"\n{city}明天有大到暴雨"
    return res    

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询当天的天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'default': '北京', 'description': '具体的城市', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明天的天气预报', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

## 模型调用工具

In [8]:
## 1.将工具绑定到模型上
tmodel = model.bind_tools([get_weather])
## 2.创建应该消息列表
msgs = [HumanMessage("杭州明天的天气任何？明天呢")]
## 3.调用模型
resp = tmodel.invoke(msgs)
## 4.把返回的消息添加到信息列表
msgs.append(resp)
## 5.获取tool_calls列表
tool_calls = resp.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather':
        # 5.1调用工具
        tmsg = get_weather.invoke(tool_call)
        msgs.append(tmsg)
## 6.把我们获取到的数据提供给模型
resp_final = tmodel.invoke(msgs)
## 7.添加到信息列表
msgs.append(resp_final)
## 8.遍历输出所有信息
for msg in msgs:
    msg.pretty_print()

================================ Human Message =================================

杭州明天的天气任何？明天呢
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_1bqqi6it)
 Call ID: call_1bqqi6it
  Args:
    city: 杭州
    if_forecast: True
================================= Tool Message =================================
Name: get_weather

杭州明天天气不错
杭州明天有大到暴雨
================================== Ai Message ==================================

杭州明天的天气情况如下：
- 白天天气总体不错
- 但傍晚至夜间有大到暴雨

建议您：
1️⃣ 出行携带雨具
2️⃣ 避免在暴雨时段外出
3️⃣ 关注实时天气预警

需要我帮您查询更详细的小时级预报吗？


## 2.写docstring

In [10]:
## 模型在上面的代码中已经创建了，我们可以继续写工具
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool

## 使用docstring发方式不需要arg_schema
@tool(parse_docstring=True)
def get_weather(city:str="上海",if_forecast:bool=False):
    """ 
    查询当天的天气，可以包含明天的天气预报

    Args:
         city : 具体的城市
         if_forecast : 是否包含明天的天气预报

    Returns:
         城市的当天的天气，可以包含明天的天气预报     
    """
    res = f"{city}明天天气不错"
    if if_forecast:
        res += f"\n{city}明天有大到暴雨"
    return res    

rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询当天的天气，可以包含明天的天气预报',
        'parameters': {
            'properties': {
                'city': {'default': '上海', 'description': '具体的城市', 'type': 'string'},
                'if_forecast': {'default': False, 'description': '是否包含明天的天气预报', 'type': 'boolean'}
            },
            'type': 'object'
        }
    }
}

In [ ]:
## 1.将工具绑定到模型上
tmodel = model.bind_tools([get_weather])
## 2.创建应该消息列表
msgs = [HumanMessage("杭州明天的天气任何？明天呢")]
## 3.调用模型
resp = tmodel.invoke(msgs)
## 4.把返回的消息添加到信息列表
msgs.append(resp)
## 5.获取tool_calls列表
tool_calls = resp.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == 'get_weather':
        # 5.1调用工具
        tmsg = get_weather.invoke(tool_call)
        msgs.append(tmsg)
## 6.把我们获取到的数据提供给模型
resp_final = tmodel.invoke(msgs)
## 7.添加到信息列表
msgs.append(resp_final)
## 8.遍历输出所有信息
for msg in msgs:
    msg.pretty_print()

## 3.多工具调用1

### 创建模型实例

In [12]:
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from model_utils import create_qwen3_instance

model = create_qwen3_instance() # 利用我们的函数来创建模型。
print(model)

metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.2.12'}} profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000002142C80BC50> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002142C94A0D0> root_client=<openai.OpenAI object at 0x000002142C80B890> root_async_client=<openai.AsyncOpenAI object at 0x000002142C949F90> model_name='qwen3-vl:latest' temperature=0.1 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='http://localhost:11434/v1'


### 上面已经有get_weather工具了，我们来创建一个get_stock_price工具

In [47]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_stock_price(company:str,timeframe: str="today") ->str:
    """ 
    获取指定公司的股票在指定时间的价格

    Args:
         company : 具体的公司名称
         timeframe : 时间范围 (today-今日,week-本周,month-本月)

    """
    print(timeframe)
    stock_data = {
        "苹果公司":{"today":185.2,"week":183.5,"month":180.75},
        "谷歌公司":{"today":15.42,"week":15.2,"month":14.85},
        "微软公司":{"today":415.86,"week":412.30,"month":405.42},
    }

    if company in stock_data:
        price = stock_data[company].get(timeframe,"")
        return f"{company} {timeframe}的股价是{price}美元"
    else:
        return f"没有找到{company}的股票信息。"
    

In [51]:
get_stock_price.invoke({"company":"微软公司","timeframe":"month"})

month


'微软公司 month的股价是405.42美元'

### 再来定义一个工具，search_new

In [55]:
from langchain_core.tools import tool

@tool(parse_docstring=True)
def search_news(company:str) ->str:
    """ 
    搜索指定公司的新闻

    Args:
         company : 公司名称，如：谷歌公司

    Returns:
         指定公司的新闻     
    """
    news_data = {
        "苹果公司":[
            "发布新款Iphone，股价上涨3%","苹果与欧盟达成反垄断和解协议","苹果将在印度扩大生产规模"
        ],

        "谷歌公司":[
           "谷歌发布AI新模型，性能提升20%","谷歌与OpenAI合作，开发AI新模型","谷歌在欧洲开展AI新研究项目",
        ],

        "微软公司":[
           "微软Azuze云业务季度增长超预期","微软完成对Nuance公司的收购","微软推出新一代AI助手Copilot"
        ],
    }

    if company in news_data:
        return "\n".join(news_data[company])
    else:
        return f"找不到关于{company}的新闻"

In [ ]:
search_news.invoke({"company":"微软公司"})

'微软Azuze云业务季度增长超预期\n微软完成对Nuance公司的收购\n微软推出新一代AI助手Copilot'

In [59]:
## 模型实例在上面的代码中创建了，并且运行了。模型在内存里面
## 绑定工具
tools = [get_stock_price,search_news]
tmodel = model.bind_tools(tools)

msg_list = []
hmsg = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻")
# hmsg = HumanMessage(content="比较一下微软公司和苹果公司的股价")
# hmsg = HumanMessage(content="腾讯公司最近有什么新闻")
# hmsg = HumanMessage(content="海水为什么是咸的？")
msg_list.append(hmsg)

## 调用工具，因为工具多于一个需要使用循环
while True:
    res = tmodel.invoke(msg_list)
    msg_list.append(res)

    ## 如果模型不需要调用工具，则直接退出循环
    if not res.tool_calls:
        break
    ## 如果有工具调用，处理工具响应
    for tool_call in res.tool_calls:
        if tool_call["name"] == "get_stock_price":
            stockmsg = get_stock_price.invoke(tool_call)
            print(f"Stcok Result:\n{stockmsg}")
            msg_list.append(stockmsg)
        if tool_call["name"] == "search_news":  
            newmsg = search_news.invoke(tool_call)  
            print(f"New Result:\n{newmsg}")
            msg_list.append(newmsg)

## 遍历消息
for msg in msg_list:
    msg.pretty_print()            

today
Stcok Result:
content='苹果公司 today的股价是185.2美元' name='get_stock_price' tool_call_id='call_sw10spbx'
New Result:
content='发布新款Iphone，股价上涨3%\n苹果与欧盟达成反垄断和解协议\n苹果将在印度扩大生产规模' name='search_news' tool_call_id='call_s8xiihyt'
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_sw10spbx)
 Call ID: call_sw10spbx
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_s8xiihyt)
 Call ID: call_s8xiihyt
  Args:
    company: 苹果公司
================================= Tool Message =================================
Name: get_stock_price

苹果公司 today的股价是185.2美元
================================= Tool Message =================================
Name: search_news

发布新款Iphone，股价上涨3%
苹果与欧盟达成反垄断和解协议
苹果将在印度扩大生产规模
================================== Ai Message ==================================

苹果公司今日股价为 **185.2美元**。  
近期重要新闻包括：  
1️⃣ 发布

## 4.多工具调用

### 定义工具

In [4]:
## 定义工具
from langchain_core.tools import tool

@tool(parse_docstring=True)
def check_weather(city:str="广州"):
    """ 
    查询具体城市的天气

    Args:
          city : 城市名称，如北京

    Returns:    
            城市当天的天气  
    """
    return f"{city}今天天气很好，阳光明媚，适合郊游~~"


@tool(parse_docstring=True)
def get_news() ->str:
    """ 
    获取新闻

    Returns:
             返回当前的热点新闻
    """
    return "最近AI智能体非常火爆，有大量的工作岗位缺口，需求量非常大!!!"

## 调用工具

In [1]:
## 实例化模型对象
from langchain.messages import HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint
from langchain_core.utils.function_calling import convert_to_openai_tool
from model_utils import create_qwen3_instance

model = create_qwen3_instance() # 利用我们的函数来创建模型。
print(model)

metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.2.12'}} profile={} client=<openai.resources.chat.completions.completions.Completions object at 0x000001D407CAFE00> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D408208980> root_client=<openai.OpenAI object at 0x000001D407CACC20> root_async_client=<openai.AsyncOpenAI object at 0x000001D4082086E0> model_name='qwen3-vl:latest' temperature=0.1 model_kwargs={} openai_api_key=SecretStr('**********') openai_api_base='http://localhost:11434/v1'


In [5]:
from rich import print as rprint
## 绑定工具
tools = [check_weather,get_news]

tmodel = model.bind_tools(tools)

msgs = [
    HumanMessage("今天杭州天气如何？今天有什么新闻？别瞎编")
]

resp = tmodel.invoke(msgs)
rprint(resp)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 170,
            'prompt_tokens': 200,
            'total_tokens': 370,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 186}
        },
        'model_provider': 'openai',
        'model_name': 'qwen3-vl:latest',
        'system_fingerprint': 'fp_ollama',
        'id': 'chatcmpl-324',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c6e5-5e6a-7ce1-8d9d-e5069c3a3048-0',
    tool_calls=[
        {'name': 'check_weather', 'args': {'city': '杭州'}, 'id': 'call_p2ges8sh', 'type': 'tool_call'},
        {'name': 'get_news', 'args': {}, 'id': 'call_kv28x2m1', 'type': 'tool_call'}
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 200,
        'output_tokens': 170,
        'total_tokens': 370,
        'input_token_details': {'cache_read': 186},
        'output_token_details': {}
    }
)

In [7]:
msgs.append(resp)
tool_calls = resp.tool_calls
for tool_call in tool_calls:
    if tool_call['name'] == "check_weather":
        check_msg = check_weather.invoke(tool_call)
        print(f"check_weather result:{check_msg}")
        msgs.append(check_msg)
    elif  tool_call['name'] =="get_news":
        news_msg = get_news.invoke(tool_call)  
        print(f"get_news result:{news_msg}") 
        msgs.append(news_msg)
    else:
        raise Exception("不存在的工具")    

final = tmodel.invoke(msgs)
msgs.append(final)    
for msg in msgs:
    msg.pretty_print()


check_weather result:content='杭州今天天气很好，阳光明媚，适合郊游~~' name='check_weather' tool_call_id='call_p2ges8sh'
get_news result:content='最近AI智能体非常火爆，有大量的工作岗位缺口，需求量非常大!!!' name='get_news' tool_call_id='call_kv28x2m1'
================================ Human Message =================================

今天杭州天气如何？今天有什么新闻？别瞎编
================================== Ai Message ==================================
Tool Calls:
  check_weather (call_p2ges8sh)
 Call ID: call_p2ges8sh
  Args:
    city: 杭州
  get_news (call_kv28x2m1)
 Call ID: call_kv28x2m1
  Args:
================================= Tool Message =================================
Name: check_weather

杭州今天天气很好，阳光明媚，适合郊游~~
================================= Tool Message =================================
Name: get_news

最近AI智能体非常火爆，有大量的工作岗位缺口，需求量非常大!!!
================================== Ai Message ==================================

杭州今天天气晴朗，阳光明媚，气温适宜，非常适合外出郊游。  
关于新闻，近期AI智能体领域热度持续攀升，相关岗位需求量大增，就业市场呈现明显增长趋势。
